# 01 — Data Prep & Sampling (multi-city)

Reconciles `points.csv` ∩ `metadata.csv` ∩ actual front-view images
present, for EVERY city in `configs/paths.yaml`'s `cities` list, for
both classes -- same three-way join as before, just looped per city
(`points.csv` still contains ids without a corresponding SVI file).

**Revised for the multi-city setup:**
- **City-prefixed `point_id` from HERE, not deferred to a later pooling
  step** (the old 2-city pipeline only prefixed ids at 05b, well after
  01-04 had already run per city separately). `point_id` is now
  `"{city_prefix}_{class}_{id}"` (e.g. `bog_positive_123`,
  `kra_negative_45`), using `configs/paths.yaml`'s `city_prefix` map --
  globally unique from this notebook onward, since every city's output
  from here through 04 lands in ONE COMBINED `interim`/`processed` tree
  (`paths.yaml`'s `interim_dir`/`processed_dir`, sibling to `base_dir`,
  not one per city).
- **No CV fold assignment here.** This branch does NOT assign
  `fold_rep{r}` columns (the old spatial K-means blocking used to live
  in this notebook) -- fold/split assignment is left to whichever later
  stage actually needs it. `src/tvg_builder.py`'s `crash_history`
  peer-matching (ablation-only, unused by the default scenarios) already
  degrades to "no peers" when no `fold_rep*` column is present, so this
  is safe as-is; anything downstream that still expects `fold_rep*`
  columns (`05`/`05b`/`07`'s formal k-fold pipeline) will need revisiting
  separately before it's run against this multi-city output.

Output: one combined `interim/reconciled_points.parquet` covering every
city -- the manifest every later notebook (`02`, `04`) iterates over.

In [ ]:
# ── Clone/update repo, mount Drive, load config ─────────────────
REPO_URL = "https://github.com/AditPradana36/crash-dualgraph.git"
REPO_DIR = "/content/crash-dualgraph"

import os
if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

import sys
sys.path.append(f"{REPO_DIR}/src")

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q pyyaml tqdm pandas

In [ ]:
import yaml
import numpy as np
from pathlib import Path

with open(f"{REPO_DIR}/configs/paths.yaml") as f:
    paths_cfg = yaml.safe_load(f)

CITIES = paths_cfg["cities"]
CITY_PREFIX = paths_cfg["city_prefix"]
INTERIM_DIR = Path(paths_cfg["interim_dir"])
INTERIM_DIR.mkdir(parents=True, exist_ok=True)

RECONCILED_OUT = INTERIM_DIR / "reconciled_points.parquet"
FORCE_RECOMPUTE = False  # set True to rebuild even if the output already exists

print(f"Cities: {CITIES}")
print(f"City prefixes: {CITY_PREFIX}")
print(f"Combined interim dir: {INTERIM_DIR}")
print(f"Output: {RECONCILED_OUT}")
print(f"Expect ~{paths_cfg['approx_n_positive']}+ positive, ~{paths_cfg['approx_n_negative']}+ negative (combined, all cities)")

In [ ]:
# ── Skip everything below if already done (checkpoint pattern) ─────────
if RECONCILED_OUT.exists() and not FORCE_RECOMPUTE:
    import pandas as pd
    reconciled = pd.read_parquet(RECONCILED_OUT)
    print(f"✅ Found existing reconciled output ({len(reconciled)} rows) — loaded, skipping recomputation.")
    print("   Set FORCE_RECOMPUTE = True above and re-run if you need to rebuild it.")
else:
    print("No existing output found (or FORCE_RECOMPUTE=True) — running full reconciliation below.")

In [ ]:
# ── Reconciliation logic (per city, per class) ─────────────────
import re
import pandas as pd
from tqdm.auto import tqdm

IMG_PATTERN = re.compile(r"^(\d+)_dashcam_fov90\.(jpg|jpeg|png)$", re.IGNORECASE)

def extract_id_from_filename(fname: str):
    m = IMG_PATTERN.match(fname)
    return int(m.group(1)) if m else None

def reconcile_class(city, city_prefix, cls_name, points_csv, metadata_csv, front_view_dir):
    pts = pd.read_csv(points_csv)
    meta = pd.read_csv(metadata_csv)

    image_ids = {}
    for p in Path(front_view_dir).iterdir():
        _id = extract_id_from_filename(p.name)
        if _id is not None:
            image_ids[_id] = str(p)

    ids_points = set(pts["id"])
    ids_meta = set(meta["id"])
    ids_images = set(image_ids.keys())

    ids_pts_not_meta = ids_points - ids_meta
    ids_meta_not_pts = ids_meta - ids_points
    ids_pm = ids_points & ids_meta
    ids_pm_not_img = ids_pm - ids_images
    ids_final = ids_pm & ids_images

    print(f"[{city}/{cls_name}]")
    print(f"  points.csv={len(ids_points)}  metadata.csv={len(ids_meta)}  images_on_disk={len(ids_images)}")
    print(f"  points ∩ metadata = {len(ids_pm)}  "
          f"(dropped: {len(ids_pts_not_meta)} points-only, {len(ids_meta_not_pts)} metadata-only)")
    print(f"  (points ∩ metadata) ∩ images = {len(ids_final)}  "
          f"(further dropped, no image file: {len(ids_pm_not_img)})")
    if ids_pm_not_img:
        sample = sorted(list(ids_pm_not_img))[:10]
        print(f"  example ids missing an image (first 10): {sample}")
    print()

    merged = pts.merge(meta, on="id", how="inner")
    merged = merged[merged["id"].isin(ids_images)].copy()
    merged["image_path"] = merged["id"].map(image_ids)
    merged["class"] = cls_name
    merged["label"] = 1 if cls_name == "positive" else 0
    merged["city"] = city
    # City-prefixed, globally unique from here on -- see notebook intro.
    merged["point_id"] = city_prefix + "_" + merged["class"] + "_" + merged["id"].astype(str)

    # Informational only -- metadata_outofrange_removed.csv implies distance
    # filtering already happened upstream; not re-applied here.
    if "status" in merged.columns:
        n_not_ok = (merged["status"] != "ok").sum()
        if n_not_ok:
            print(f"  ⚠️  {n_not_ok} reconciled rows have status != 'ok' — "
                  f"unexpected given the filename, worth a manual check.")
    return merged

In [ ]:
if not (RECONCILED_OUT.exists() and not FORCE_RECOMPUTE):
    reconciled_parts = []
    for city in tqdm(CITIES, desc="Reconciling cities"):
        city_cfg = paths_cfg["per_city"][city]
        prefix = CITY_PREFIX[city]
        class_specs = [
            ("positive", city_cfg["positive_points_csv"], city_cfg["positive_metadata_csv"], city_cfg["positive_front_view_dir"]),
            ("negative", city_cfg["negative_points_csv"], city_cfg["negative_metadata_csv"], city_cfg["negative_front_view_dir"]),
        ]
        for cls_name, pts_csv, meta_csv, fv_dir in class_specs:
            reconciled_parts.append(reconcile_class(city, prefix, cls_name, pts_csv, meta_csv, fv_dir))

    reconciled = pd.concat(reconciled_parts, ignore_index=True)
    assert reconciled["point_id"].is_unique, (
        "city-prefixed point_id collided across cities -- shouldn't be possible, investigate.")
    print(f"Combined reconciled dataset: {len(reconciled)} rows "
          f"({(reconciled['label']==1).sum()} positive, {(reconciled['label']==0).sum()} negative)")
    display(reconciled.groupby("city")["label"].agg(["count", "sum"]).rename(columns={"sum": "n_positive"}))

In [ ]:
# ── Save checkpoint ────────────────────────────────
if not (RECONCILED_OUT.exists() and not FORCE_RECOMPUTE):
    reconciled.to_parquet(RECONCILED_OUT, index=False)
    print(f"✅ Saved {len(reconciled)} reconciled points ({len(CITIES)} cities) to {RECONCILED_OUT}")

print()
print(reconciled[["point_id", "city", "class", "label", "highway", "input_lat", "input_lon"]].head(10))
print()
print("Next: 02_svi_segmentation.ipynb")